<a href="https://colab.research.google.com/github/ojassahu29/Neural-Style-Transfer-/blob/main/NeuralStyleTransfer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Neural Style Transfer using VGG-19

In this, there will be application of the artistic style of one image (like a painting) to the content of another image (like a photo) using a pre-trained **VGG-19** convolutional neural network.

---

## Project Objectives

- Load and preprocess content and style images
- Extract style and content features using **VGG-19**
- Define style and content loss functions
- Optimize an image to combine structure from the content and texture from the style

---

## Importing Required Libraries and Configuring Warnings


- **TensorFlow / Keras** – Core deep learning framework
- **VGG-19** – Pre-trained CNN for feature extraction
- **NumPy + Matplotlib** – Image manipulation and visualization
- **PIL** – Handling image formats and resizing
- **ipywidgets** – Interactive UI (dropdowns, buttons) in Colab
- **Google Colab** – Free GPU acceleration and notebook interface
- `warnings.filterwarnings(...)` to hide non-critical Keras messages since the messages were clogging up lots of space in the output

# Ojas Sahu (2023A3PS0861G)

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import time
import functools
import warnings
import ipywidgets as widgets
warnings.filterwarnings("ignore", category=UserWarning)

from tensorflow.keras.applications import vgg19
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing import image as kpimage
from tensorflow.keras import backend as K
from google.colab import files
from IPython.display import display
from PIL import Image

## Image Preprocessing and Conversion Functions

This block defines two utility functions:

###  `load_img(path_of_image, max_dim=512)`
- Loads an image from the given file path
- Decodes and normalizes it to a float32 tensor
- Resizes it while preserving the aspect ratio (maximum dimension = 512 by default)

###  `tensor_to_image(tensor)`
- Converts the final stylized tensor (output of the model) back into an image
- Clips the values to `[0, 255]` and casts it to `uint8` format
- Removes the batch dimension to return a standard image

These helper functions are essential for preparing inputs and visualizing outputs of the NST pipeline.


In [ ]:
def load_img(path_of_image, max_dim=512):
  image = tf.io.read_file(path_of_image)
  image = tf.image.decode_image(image, channels=3)
  image = tf.image.convert_image_dtype(image, tf.float32)
  shape = tf.cast(tf.shape(image)[:-1], tf.float32)
  scale = max_dim / max(shape)
  shape1 = tf.cast(shape*scale, tf.int32)
  image = tf.image.resize(image, shape1)
  image = image[tf.newaxis, :]
  return image

def tensor_to_image(tensor):
  tensor = tensor*255
  tensor = tf.clip_by_value(tensor, 0, 255)
  tensor = tf.cast(tensor, tf.uint8)
  return tensor[0]

## Feature Extraction and Style Representation



This block defines core components for extracting and representing content and style:

---

### `content_layers` & `style_layers`
- Specifies the VGG-19 convolutional layers used to extract content and style features.
- Content Layer:
  - `block4_conv2` captures high-level structural features of the image.
- Style Layers:
  - From `block1_conv1` to `block5_conv1`, progressively capture low-level to high-level texture patterns.

---

### `vgg_layers(layer_name)`
- Returns a custom VGG-19 model that outputs feature maps from specified intermediate layers.
- The pre-trained VGG-19 model is loaded without its fully connected top layers (`include_top=False`) and is set to non-trainable.

---

### `gram_matrix(input_tensor)`
- Computes the **Gram Matrix** of an input tensor.
- The Gram matrix captures **style** by computing correlations between feature maps.
- This representation is used to calculate the **style loss** between the generated image and the style image.


In [ ]:
content_layers = ['block4_conv2']
style_layers = ['block1_conv1', 'block2_conv1', 'block3_conv1', 'block4_conv1', 'block5_conv1']
num_content_layers = len(content_layers)
num_style_layers = len(style_layers)

In [ ]:
def vgg_layers(layer_name):
  #Creates a VGG19 model that returns a list of intermediate values (output)
  vgg = vgg19.VGG19(include_top=False, weights='imagenet')
  vgg.trainable = False #feature extractor
  outputs = [vgg.get_layer(name).output for name in layer_name]
  model = tf.keras.Model([vgg.input], outputs)
  return model

In [ ]:
def gram_matrix(input_tensor):
    result = tf.linalg.einsum('bijc,bijd->bcd', input_tensor, input_tensor)
    input_shape = tf.shape(input_tensor)
    num_locations = tf.cast(input_shape[1] * input_shape[2], tf.float32)
    return result / num_locations

## StyleContentModel – Extracting Style and Content Features



This class extends `tf.keras.models.Model` and wraps the VGG-19 model to extract content and style features from input images.

---

###  `__init__(...)`
- Initializes a custom VGG-19 model using the specified `style_layers` and `content_layers`.
- Combines these layers into one model using the `vgg_layers()` function.
- Sets `self.vgg.trainable = False` to freeze all VGG parameters (no backprop to VGG).

---

###  `call(inputs)`
- Preprocesses the input using VGG-19's built-in preprocessing:
  - Scales from `[0, 1]` to `[0, 255]`
  - Applies mean subtraction and channel reordering (RGB → BGR)
- Passes the image through VGG and separates:
  - **Style outputs** → Passed through `gram_matrix()` to capture texture
  - **Content outputs** → Left as is to capture structure
- Returns a dictionary with:
  - `{'content': ..., 'style': ...}` for easy loss calculation later

---

This model is used during optimization to compare generated image features with those of the content and style reference images.


In [ ]:
class StyleContentModel(tf.keras.models.Model):
    def __init__(self, style_layers, content_layers):
        super().__init__() #initalize parent keras model
        self.vgg = vgg_layers(style_layers + content_layers)
        self.style_layers = style_layers
        self.content_layers = content_layers
        self.vgg.trainable = False

    def call(self, inputs):
        inputs = inputs * 255.0
        preprocessed_input = vgg19.preprocess_input(inputs)
        outputs = self.vgg.call(preprocessed_input)

        style_outputs, content_outputs = (
            outputs[:num_style_layers], outputs[num_style_layers:])

        style_outputs = [gram_matrix(style_output) for style_output in style_outputs]

        content_dict = {content_name: value
                        for content_name, value
                        in zip(self.content_layers, content_outputs)}

        style_dict = {style_name: value
                      for style_name, value
                      in zip(self.style_layers, style_outputs)}

        return {'content': content_dict, 'style': style_dict}

In [ ]:
def style_content_loss(outputs, style_targets, content_targets,
                       style_weight=1e2, content_weight=1e4):  # increased style weight
    style_outputs = outputs['style']
    content_outputs = outputs['content']

    style_loss = tf.add_n([
        tf.reduce_mean((style_outputs[name] - style_targets[name]) ** 2)
        for name in style_outputs
    ])
    style_loss *= style_weight / len(style_outputs)

    content_loss = tf.add_n([
        tf.reduce_mean((content_outputs[name] - content_targets[name]) ** 2)
        for name in content_outputs
    ])
    content_loss *= content_weight / len(content_outputs)

    total_loss = style_loss + content_loss
    return total_loss


## train_step – One Optimization Iteration

This function performs a single training step for neural style transfer. It optimizes the generated image to reduce both style and content loss.

---

### Parameters:
- `image`: The current generated image (as a trainable `tf.Variable`)
- `extractor`: An instance of `StyleContentModel`, used to extract intermediate features
- `style_targets`: Style features from the style image
- `content_targets`: Content features from the content image
- `opt`: The optimizer (e.g., Adam) used to update the image

In [ ]:
def train_step(image, extractor, style_targets, content_targets, opt):
    with tf.GradientTape() as tape:
        outputs = extractor(image)
        loss = style_content_loss(outputs, style_targets, content_targets)
    grad = tape.gradient(loss, image)
    opt.apply_gradients([(grad, image)])
    image.assign(tf.clip_by_value(image, 0.0, 1.0))

##  run_style_transfer – Full Stylization Pipeline

This function performs the full neural style transfer by optimizing a generated image to combine the **content** of one image with the **style** of another.

---

### Parameters:
- `content_path`: Path to the content image file
- `style_path`: Path to the style image file
- `epochs`: Number of full passes through the optimization loop (default: 5)
- `steps_per_epoch`: Number of gradient steps per epoch (default: 100)

---

### What It Does:
1. Creates a `StyleContentModel` to extract intermediate features
2. Loads and extracts style features from the style image
3. Loads and extracts content features from the content image
4. Initializes the **generated image** as a trainable copy of the content image
5. Optimizes the generated image using Adam to minimize combined **style + content loss**
6. Clips the image after each update to keep pixel values valid (`[0, 1]`)
7. Prints progress after each epoch

---

###  Returns:
- The final stylized image as a TensorFlow tensor



In [ ]:
def run_style_transfer(content_path, style_path, epochs=5, steps_per_epoch=100):
    extractor = StyleContentModel(
        style_layers=[
            'block1_conv1', 'block2_conv1',
            'block3_conv1', 'block4_conv1', 'block5_conv1'
        ],
        content_layers=['block4_conv2']
    )

    style_targets = extractor(load_img(style_path))['style']
    content_targets = extractor(load_img(content_path))['content']

    image = tf.Variable(load_img(content_path))  # the generated image

    opt = tf.optimizers.Adam(learning_rate=0.02)

    for n in range(epochs):
        for m in range(steps_per_epoch):
            with tf.GradientTape() as tape:
                outputs = extractor(image)
                loss = style_content_loss(outputs, style_targets, content_targets)

            grad = tape.gradient(loss, image)
            opt.apply_gradients([(grad, image)])
            image.assign(tf.clip_by_value(image, 0.0, 1.0))

        print(f"Epoch {n+1} / {epochs} complete")

    return image


##  Interactive NST UI – Dropdown Selection & Execution

This block provides a simple interactive UI for running Neural Style Transfer with custom content and style images.

---

###  Features:
- Upload multiple image files at once
- Use dropdown menus to manually assign:
  -  Content image
  -  Style image
- Display both selected images as previews
- Click a **Run Style Transfer** button to:
  - Execute the NST pipeline
  - Visualize the stylized result

---

###  Highlights:
- Uses `ipywidgets` for a clean dropdown and button interface
- Disables PIL decompression bomb warning for large images
- Automatically scales image previews to fit the screen
- Results are displayed below the button after execution

 This makes it easy to experiment with different image combinations and styling parameters directly in Colab.


In [ ]:
# Upload content and style images
uploaded = files.upload()

image_paths = list(uploaded.keys())

content_dropdown = widgets.Dropdown(options=image_paths, description='Content Image:')
style_dropdown = widgets.Dropdown(options=image_paths, description='Style Image:')
run_button = widgets.Button(description="Run Style Transfer", button_style='success')
output_box = widgets.Output()

# Resize preview and disable PIL image bomb warnings
Image.MAX_IMAGE_PIXELS = None

def show_image(img_path, title):
    with Image.open(img_path) as img:
        img.thumbnail((512, 512))
        plt.imshow(img)
        plt.axis('off')
        plt.title(title)
        plt.show()

# What to do when button is clicked
def on_button_click(b):
    output_box.clear_output()
    with output_box:
        content_path = content_dropdown.value
        style_path = style_dropdown.value

        print("Content:", content_path)
        print("Style:", style_path)

        show_image(content_path, "Content Image")
        show_image(style_path, "Style Image")

        print("Running Style Transfer")
        output_image = run_style_transfer(content_path, style_path, epochs=5, steps_per_epoch=50)

        print("Stylized Output:")
        plt.imshow(tensor_to_image(output_image))
        plt.axis('off')
        plt.title("Stylized Result")
        plt.show()

# Link button to function
run_button.on_click(on_button_click)

# Display interface
display(content_dropdown, style_dropdown, run_button, output_box)


Saving cat.jpg to cat (2).jpg
Saving a.jpg to a (3).jpg


Dropdown(description='Content Image:', options=('cat (2).jpg', 'a (3).jpg'), value='cat (2).jpg')

Dropdown(description='Style Image:', options=('cat (2).jpg', 'a (3).jpg'), value='cat (2).jpg')

Button(button_style='success', description='Run Style Transfer', style=ButtonStyle())

Output()

Ended::

